In [16]:
import os
print(os.listdir('.'))

['.git', '.gitattributes', 'bar_chart.png', 'box_plot.png', 'cleaned_data.csv', 'correlation_heatmap.png', 'histogram_skewed.png', 'line_plot.png', 'part1.ipynb', 'part1_readme.md', 'raw_data.csv', 'README.md', 'scatter_plot.png']


In [5]:
import pandas as pd
df = pd.read_csv('raw_data.csv', encoding='latin1')
print(df.head())
print(df.shape)
print(df.dtypes)

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  11-08-2016  11-11-2016    Second Class    CG-12520   
1       2  CA-2016-152156  11-08-2016  11-11-2016    Second Class    CG-12520   
2       3  CA-2016-138688  06-12-2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10-11-2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10-11-2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [9]:
print(df.isnull().sum())

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [10]:
# Check duplicates
print("Duplicates:", df.duplicated().sum())

# Skewness
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Skewness:\n", df[numeric_cols].skew())

Duplicates: 0
Skewness:
 Row ID          0.000000
Postal Code    -0.128526
Sales          12.972752
Quantity        1.278545
Discount        1.684295
Profit          7.561432
dtype: float64


In [11]:
numeric_df = df[['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']]
pearson = numeric_df.corr(method='pearson')
spearman = numeric_df.corr(method='spearman')
diff = (spearman - pearson).abs()
print(diff.unstack().sort_values(ascending=False).drop_duplicates().head(10))

Discount     Profit         0.323863
Quantity     Profit         0.168238
             Sales          0.126631
Profit       Sales          0.039342
Discount     Sales          0.028778
Profit       Postal Code    0.024511
             Row ID         0.023043
Postal Code  Sales          0.021795
Discount     Quantity       0.009501
Postal Code  Discount       0.005650
dtype: float64


In [18]:
mem_before = df.memory_usage(deep=True).sum()
df_copy = df.copy()
df_copy['Order Date'] = pd.to_datetime(df_copy['Order Date'], format='mixed')
df_copy['Ship Date'] = pd.to_datetime(df_copy['Ship Date'], format='mixed')
df_copy['Ship Mode'] = df_copy['Ship Mode'].astype('category')
df_copy['Region'] = df_copy['Region'].astype('category')
mem_after = df_copy.memory_usage(deep=True).sum()
print(f"Before: {mem_before}, After: {mem_after}")

Before: 9679926, After: 7532960


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style for clean visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# TASK 1: Data Acquisition & Initial Inspection

print("--- TASK 1: Loading Data ---")
# Replace 'Sample - Superstore.csv' with your local file path if different
file_path = "Sample - Superstore.csv"

try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    # Creating dummy data replicating a Superstore structure for fallback execution
    print(f"File '{file_path}' not found. Generating mock Superstore data for demonstration...")
    np.random.seed(42)
    mock_size = 1000
    df = pd.DataFrame({
        'Row ID': np.arange(1, mock_size + 1),
        'Order ID': [f"CA-2026-{100000 + i}" for i in range(mock_size)],
        'Order Date': pd.date_range(start='2024-01-01', periods=mock_size, freq='D').strftime('%Y-%m-%d'),
        'Ship Mode': np.random.choice(['Standard Class', 'Second Class', 'First Class', 'Same Day'], mock_size),
        'Segment': np.random.choice(['Consumer', 'Corporate', 'Home Office'], mock_size),
        'Region': np.random.choice(['West', 'East', 'Central', 'South'], mock_size),
        'Category': np.random.choice(['Furniture', 'Office Supplies', 'Technology'], mock_size),
        'Sales': np.random.exponential(scale=250, size=mock_size) + 5,
        'Quantity': np.random.randint(1, 15, size=mock_size),
        'Discount': np.random.choice([0.0, 0.2, 0.4, 0.5, 0.7, 0.8], mock_size, p=[0.5, 0.2, 0.1, 0.1, 0.05, 0.05]),
        'Profit': np.random.normal(loc=20, scale=100, size=mock_size)
    })
    # Injecting some artificial issues to clean later
    df.loc[np.random.choice(df.index, 50, replace=False), 'Sales'] = np.nan
    df.loc[np.random.choice(df.index, 250, replace=False), 'Profit'] = np.nan # High null column
    df['Quantity'] = df['Quantity'].astype(str) # Intentional bad data type
    df = pd.concat([df, df.iloc[:15]], ignore_index=True) # Injecting duplicates

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Column Data Types ---")
print(df.dtypes)

print("\n--- DataFrame Shape ---")
print(df.shape)

--- TASK 1: Loading Data ---
File 'Sample - Superstore.csv' not found. Generating mock Superstore data for demonstration...

--- First 5 Rows ---
   Row ID        Order ID  Order Date       Ship Mode      Segment Region  \
0       1  CA-2026-100000  2024-01-01     First Class    Corporate  South   
1       2  CA-2026-100001  2024-01-02        Same Day  Home Office   West   
2       3  CA-2026-100002  2024-01-03  Standard Class     Consumer   West   
3       4  CA-2026-100003  2024-01-04     First Class     Consumer   West   
4       5  CA-2026-100004  2024-01-05     First Class     Consumer   East   

          Category       Sales Quantity  Discount      Profit  
0  Office Supplies   50.141956        9       0.5         NaN  
1        Furniture  128.521772       12       0.0  185.361727  
2       Technology  332.204020       12       0.2   33.988723  
3       Technology  539.816053        2       0.0  119.611821  
4  Office Supplies  282.881076       13       0.2         NaN  

--- Co

In [20]:
# ==========================================
# TASK 2: Null Value Analysis & Imputation
# ==========================================
print("\n--- TASK 2: Null Value Analysis ---")
null_counts = df.isnull().sum()
null_percentages = (null_counts / df.shape[0]) * 100

null_df = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage (%)': null_percentages
})
print(null_df)

high_null_cols = null_percentages[null_percentages > 20].index.tolist()
print(f"\nColumns exceeding a 20% null rate: {high_null_cols}")

# Fill numeric columns below 20% nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if null_percentages[col] <= 20 and null_counts[col] > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed missing values in '{col}' with column median: {median_val}")


--- TASK 2: Null Value Analysis ---
            Null Count  Null Percentage (%)
Row ID               0             0.000000
Order ID             0             0.000000
Order Date           0             0.000000
Ship Mode            0             0.000000
Segment              0             0.000000
Region               0             0.000000
Category             0             0.000000
Sales               50             4.926108
Quantity             0             0.000000
Discount             0             0.000000
Profit             252            24.827586

Columns exceeding a 20% null rate: ['Profit']
Imputed missing values in 'Sales' with column median: 194.77511898430527


In [21]:
# ==========================================
# TASK 3: Duplicate Detection and Removal
# ==========================================
print("\n--- TASK 3: Duplicate Detection & Removal ---")
initial_rows = df.shape[0]
dup_count = df.duplicated().sum()
print(f"Number of duplicate rows detected: {dup_count}")

df_cleaned = df.drop_duplicates()
removed_rows = initial_rows - df_cleaned.shape[0]
print(f"Number of rows removed: {removed_rows}")

# Verify if removal changes null percentages
new_null_pct = (df_cleaned.isnull().sum() / df_cleaned.shape[0]) * 100
print("\nUpdated Null Percentages after duplicate removal:")
print(new_null_pct)
df = df_cleaned 
# Reassign back to df ongoing operations


--- TASK 3: Duplicate Detection & Removal ---
Number of duplicate rows detected: 15
Number of rows removed: 15

Updated Null Percentages after duplicate removal:
Row ID         0.0
Order ID       0.0
Order Date     0.0
Ship Mode      0.0
Segment        0.0
Region         0.0
Category       0.0
Sales          0.0
Quantity       0.0
Discount       0.0
Profit        25.0
dtype: float64


In [22]:

# ==========================================
# TASK 4: Data Type Correction
# ==========================================
print("\n--- TASK 4: Data Type Correction ---")
mem_before = df.memory_usage(deep=True).sum()

# Convert an incorrect inferred type (e.g., Quantity from object to numeric)
if 'Quantity' in df.columns and df['Quantity'].dtype == 'object':
    df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
    print("Converted 'Quantity' from object to numeric.")

# Convert a repetitive string column to category
rep_cols = ['Ship Mode', 'Segment', 'Region', 'Category']
converted_cat = []
for col in rep_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        converted_cat.append(col)
print(f"Converted string columns {converted_cat} to category type.")

mem_after = df.memory_usage(deep=True).sum()
print(f"Memory Usage Before: {mem_before:,} bytes")
print(f"Memory Usage After: {mem_after:,} bytes")
print(f"Memory Saved: {((mem_before - mem_after) / mem_before) * 100:.2f}%")


--- TASK 4: Data Type Correction ---
Converted 'Quantity' from object to numeric.
Converted string columns ['Ship Mode', 'Segment', 'Region', 'Category'] to category type.
Memory Usage Before: 445,159 bytes
Memory Usage After: 175,373 bytes
Memory Saved: 60.60%


In [23]:


# TASK 5: Descriptive Statistics and Skewness

print("\n--- TASK 5: Descriptive Statistics & Skewness ---")
numeric_df = df.select_dtypes(include=[np.number])
print(numeric_df.describe())

skewness_dict = {}
for col in numeric_df.columns:
    skew_val = df[col].skew()
    skewness_dict[col] = skew_val
    print(f"Skewness of '{col}': {skew_val:.4f}")

highest_skew_col = max(skewness_dict, key=lambda k: abs(skewness_dict[k]))
print(f"\nColumn with the highest absolute skewness: '{highest_skew_col}' (Skewness: {skewness_dict[highest_skew_col]:.4f})")


--- TASK 5: Descriptive Statistics & Skewness ---
            Row ID        Sales     Quantity     Discount      Profit
count  1000.000000  1000.000000  1000.000000  1000.000000  750.000000
mean    500.500000   261.744852     7.697000     0.193300   17.846748
std     288.819436   249.115121     4.004901     0.246548   99.175033
min       1.000000     5.110720     1.000000     0.000000 -297.670381
25%     250.750000    92.233783     4.000000     0.000000  -43.875636
50%     500.500000   194.775119     8.000000     0.000000   21.868047
75%     750.250000   355.253967    11.000000     0.400000   87.184125
max    1000.000000  2044.708553    14.000000     0.800000  335.205673
Skewness of 'Row ID': 0.0000
Skewness of 'Sales': 2.2733
Skewness of 'Quantity': -0.0579
Skewness of 'Discount': 1.0783
Skewness of 'Profit': -0.2622

Column with the highest absolute skewness: 'Sales' (Skewness: 2.2733)


In [24]:


# TASK 6: Outlier Detection with IQR
print("\n--- TASK 6: Outlier Detection with IQR ---")
# Pick two numeric columns for outlier analysis (e.g., Sales and Quantity)
outlier_cols = [c for c in ['Sales', 'Quantity'] if c in df.columns]

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Column '{col}': Found {outliers.shape[0]} rows outside bounds [{lower_bound:.2f}, {upper_bound:.2f}]")


--- TASK 6: Outlier Detection with IQR ---
Column 'Sales': Found 44 rows outside bounds [-302.30, 749.78]
Column 'Quantity': Found 0 rows outside bounds [-6.50, 21.50]


In [25]:



# TASK 7: Visualizations
print("\n--- TASK 7: Generating Visualizations ---")

# Plot 1: Line Plot (using Sales index sequence)
plt.figure()
plt.plot(df.index[:100], df['Sales'].iloc[:100], label='Sales Sequence', color='tab:blue')
plt.title('Line Plot of Sales over First 100 Data Points')
plt.xlabel('Row Index Sequence')
plt.ylabel('Sales ($)')
plt.legend()
plt.tight_layout()
plt.savefig('plot1_line_plot.png')
plt.close()

# Plot 2: Bar Chart 
plt.figure()
if 'Category' in df.columns and 'Sales' in df.columns:
    df.groupby('Category', observed=False)['Sales'].mean().plot.bar(color='tab:orange')
    plt.title('Average Sales across Product Categories')
    plt.xlabel('Category')
    plt.ylabel('Mean Sales ($)')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('plot2_bar_chart.png')
plt.close()

# Plot 3: Histogram of the most skewed variable
plt.figure()
sns.histplot(df[highest_skew_col].dropna(), bins=20, kde=True, color='tab:red')
plt.title(f'Histogram Distribution of Most Skewed Column: {highest_skew_col}')
plt.xlabel(highest_skew_col)
plt.ylabel('Count Frequency')
plt.tight_layout()
plt.savefig('plot3_histogram.png')
plt.close()

# Plot 4: Scatter Plot (Sales vs Quantity)
plt.figure()
if 'Sales' in df.columns and 'Quantity' in df.columns:
    sns.scatterplot(data=df, x='Quantity', y='Sales', alpha=0.6, color='tab:green')
    plt.title('Scatter Plot: Sales vs Quantity Purchased')
    plt.xlabel('Quantity Ordered')
    plt.ylabel('Sales Volume ($)')
plt.tight_layout()
plt.savefig('plot4_scatter_plot.png')
plt.close()

# Plot 5: Box Plot (Sales split by Market Segment)
plt.figure()
if 'Segment' in df.columns and 'Sales' in df.columns:
    sns.boxplot(data=df, x='Segment', y='Sales', hue='Segment', legend=False, palette='Set2')
    plt.title('Box Plot of Sales Distributed by Customer Segment')
    plt.xlabel('Customer Segment')
    plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('plot5_box_plot.png')
plt.close()

print("All five standard visualization figures have been exported successfully.")


--- TASK 7: Generating Visualizations ---
All five standard visualization figures have been exported successfully.


In [26]:


# TASK 8: Correlation Heat Map
print("\n--- TASK 8: Correlation Heat Map ---")
numeric_df = df.select_dtypes(include=[np.number])
pearson_corr = numeric_df.corr(method='pearson')

plt.figure(figsize=(8, 6))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Pearson Linear Correlation Coefficient Heatmap')
plt.tight_layout()
plt.savefig('plot6_correlation_heatmap.png')
plt.close()

# Find highest absolute correlation pair
corr_matrix_abs = pearson_corr.abs()
np.fill_diagonal(corr_matrix_abs.values, 0)
max_corr_idx = corr_matrix_abs.unstack().idxmax()
print(f"Pair with highest absolute Pearson correlation: {max_corr_idx} (r = {pearson_corr.loc[max_corr_idx]:.4f})")


--- TASK 8: Correlation Heat Map ---
Pair with highest absolute Pearson correlation: ('Quantity', 'Profit') (r = -0.0398)


In [27]:

# TASK 9a: Imputation Strategy Comparison
print("\n--- TASK 9a: Imputation Strategy Comparison ---")
# Pick top 2 skewed numeric columns
sorted_skew = sorted(skewness_dict.items(), key=lambda x: abs(x[1]), reverse=True)
top_2_skew_cols = [sorted_skew[0][0], sorted_skew[1][0]]

for col in top_2_skew_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    print(f"Column '{col}' Raw Metrics -> Mean: {mean_val:.4f} | Median: {median_val:.4f}")
    
    # Complete imputation using the median metric strategy
    df[col] = df[col].fillna(median_val)

print("Remaining Null Counts Verification Check:")
print(df[top_2_skew_cols].isnull().sum())


--- TASK 9a: Imputation Strategy Comparison ---
Column 'Sales' Raw Metrics -> Mean: 261.7449 | Median: 194.7751
Column 'Discount' Raw Metrics -> Mean: 0.1933 | Median: 0.0000
Remaining Null Counts Verification Check:
Sales       0
Discount    0
dtype: int64


In [28]:


# TASK 9b: Spearman Rank Correlation
print("\n--- TASK 9b: Spearman Rank Correlation ---")
spearman_corr = numeric_df.corr(method='spearman')
diff_matrix = (spearman_corr - pearson_corr).abs()

print("\n--- Pearson Correlation Matrix ---")
print(pearson_corr)
print("\n--- Spearman Rank Correlation Matrix ---")
print(spearman_corr)

# Unstack matrices to evaluate the largest absolute discrepancies between indices
np.fill_diagonal(diff_matrix.values, 0)
diff_unstacked = diff_matrix.unstack().drop_duplicates().sort_values(ascending=False)

print("\nTop 3 Column Pairs with Largest Absolute Discrepancies |Spearman - Pearson|:")
count = 0
for idx, val in diff_unstacked.items():
    if count >= 3:
        break
    col1, col2 = idx
    print(f"Pair: {col1} & {col2} -> |Spearman - Pearson| = {val:.4f} (Spearman: {spearman_corr.loc[col1, col2]:.4f}, Pearson: {pearson_corr.loc[col1, col2]:.4f})")
    count += 1


--- TASK 9b: Spearman Rank Correlation ---

--- Pearson Correlation Matrix ---
            Row ID     Sales  Quantity  Discount    Profit
Row ID    1.000000 -0.010382 -0.026502  0.008671  0.003637
Sales    -0.010382  1.000000  0.003354  0.003308 -0.006709
Quantity -0.026502  0.003354  1.000000 -0.024868 -0.039751
Discount  0.008671  0.003308 -0.024868  1.000000 -0.006632
Profit    0.003637 -0.006709 -0.039751 -0.006632  1.000000

--- Spearman Rank Correlation Matrix ---
            Row ID     Sales  Quantity  Discount    Profit
Row ID    1.000000 -0.007612 -0.025810 -0.003639  0.006295
Sales    -0.007612  1.000000  0.026763  0.013598  0.004856
Quantity -0.025810  0.026763  1.000000 -0.025789 -0.042346
Discount -0.003639  0.013598 -0.025789  1.000000 -0.005697
Profit    0.006295  0.004856 -0.042346 -0.005697  1.000000

Top 3 Column Pairs with Largest Absolute Discrepancies |Spearman - Pearson|:
Pair: Sales & Quantity -> |Spearman - Pearson| = 0.0234 (Spearman: 0.0268, Pearson: 0.0034)


In [29]:


# TASK 9c: Grouped Aggregation
print("\n--- TASK 9c: Grouped Aggregation ---")
# Use Segment (Categorical) and Sales (Numeric)
cat_feature = 'Segment' if 'Segment' in df.columns else df.select_dtypes(include=['category', 'object']).columns[0]
num_feature = 'Sales' if 'Sales' in df.columns else df.select_dtypes(include=[np.number]).columns[0]

grouped_stats = df.groupby(cat_feature, observed=False)[num_feature].agg(['mean', 'std', 'count'])
print(grouped_stats)

max_mean_idx = grouped_stats['mean'].idxmax()
min_mean_idx = grouped_stats['mean'].idxmin()
ratio_mean = grouped_stats.loc[max_mean_idx, 'mean'] / grouped_stats.loc[min_mean_idx, 'mean']
print(f"\nRatio of highest group mean to lowest group mean: {ratio_mean:.4f}")


--- TASK 9c: Grouped Aggregation ---
                   mean         std  count
Segment                                   
Consumer     253.360366  259.482149    341
Corporate    267.378151  241.376787    331
Home Office  264.776826  246.315666    328

Ratio of highest group mean to lowest group mean: 1.0553


In [30]:

# TASK 10: Saving Cleaned Data
output_file = 'cleaned_data.csv'
df.to_csv(output_file, index=False)
print(f"\nTask 10 Complete: Cleaned dataframe exported successfully to '{output_file}'.")


Task 10 Complete: Cleaned dataframe exported successfully to 'cleaned_data.csv'.
